In [ ]:
# process LCMS data- exported as mzData.xml from Agilent Mass Hunter Software

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import xml.etree.ElementTree as ET
import base64
import struct
import zlib
from scipy.interpolate import interp1d

# Place your data in folders relative to this script
INPUT_DIRECTORY = "./data/input" # Folder with .mzdata.xml files

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.linewidth'] = 1.5
plt.rcParams['lines.linewidth'] = 1.5

def decode_binary_data(encoded_data, precision=32, endian='little', compression=None):
    """Decode base64 encoded binary data."""
    try:
        binary = base64.b64decode(encoded_data)
        
        if compression == 'zlib':
            binary = zlib.decompress(binary)
        
        endian_char = '<' if endian == 'little' else '>'
        
        if precision == 64:
            dtype = f'{endian_char}f8'
        else:
            dtype = f'{endian_char}f4'
        
        data = np.frombuffer(binary, dtype=dtype)
        return data
    
    except Exception as e:
        print(f"        Decode error: {e}")
        return np.array([])

def get_clean_filename(filepath):
    """Extract clean filename without .mzdata extension."""
    stem = Path(filepath).stem
    clean_name = stem.replace('.mzdata', '')
    return clean_name

def read_mzdata_file(filepath):
    """Read mzData XML - extracting data from <data> elements inside arrays.
    
    Retention time is calculated as: RT (min) = scan_index * (10 / 900)
    This is based on the instrument info that 900 scans = 10 minutes.
    """
    ms1_scans = []
    ms1_times = []
    ms2_scans = []
    
    print(f"\n  Reading: {Path(filepath).name}")
    
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()
        
        spectra = root.findall('.//{*}spectrum')
        print(f"  Found {len(spectra)} spectra")
        
        # Retention time calculation: 900 scans = 10 minutes
        time_per_scan = 10.0 / 900.0  # ~0.01111 minutes per scan
        
        for idx, spectrum in enumerate(spectra):
            try:
                # Get MS level from spectrumInstrument
                ms_level = '1'
                spec_instr = spectrum.find('.//{*}spectrumInstrument')
                if spec_instr is not None:
                    ms_level = spec_instr.get('msLevel', '1')
                
                # Calculate retention time: scan index * time per scan
                rt = idx * time_per_scan
                
                parent_mz = 0
                precursor = spectrum.find('.//{*}precursorMZ')
                if precursor is not None and precursor.text:
                    try:
                        parent_mz = float(precursor.text)
                    except:
                        pass
                
                mz_array_binary = spectrum.find('{*}mzArrayBinary')
                intensity_array_binary = spectrum.find('{*}intenArrayBinary')
                
                mz_data = None
                intensity_data = None
                
                if mz_array_binary is not None and intensity_array_binary is not None:
                    mz_data_elem = mz_array_binary.find('{*}data')
                    if mz_data_elem is None:
                        mz_data_elem = mz_array_binary.find('data')
                    
                    intensity_data_elem = intensity_array_binary.find('{*}data')
                    if intensity_data_elem is None:
                        intensity_data_elem = intensity_array_binary.find('data')
                    
                    if mz_data_elem is not None and intensity_data_elem is not None:
                        mz_precision = int(mz_data_elem.get('precision', 32))
                        mz_endian = mz_data_elem.get('endian', 'little')
                        mz_compression = mz_data_elem.get('compression', None)
                        mz_text = (mz_data_elem.text or '').strip()
                        
                        intensity_precision = int(intensity_data_elem.get('precision', 32))
                        intensity_endian = intensity_data_elem.get('endian', 'little')
                        intensity_compression = intensity_data_elem.get('compression', None)
                        intensity_text = (intensity_data_elem.text or '').strip()
                        
                        if mz_text and intensity_text:
                            mz_data = decode_binary_data(
                                mz_text,
                                mz_precision,
                                mz_endian,
                                mz_compression
                            )
                            intensity_data = decode_binary_data(
                                intensity_text,
                                intensity_precision,
                                intensity_endian,
                                intensity_compression
                            )
                
                if mz_data is not None and intensity_data is not None:
                    if len(mz_data) > 0 and len(intensity_data) > 0:
                        if ms_level == '1':
                            ms1_scans.append({
                                'mz': mz_data,
                                'intensity': intensity_data
                            })
                            ms1_times.append(rt)
                        else:
                            ms2_scans.append({
                                'mz': mz_data,
                                'intensity': intensity_data,
                                'parent_mz': parent_mz
                            })
            
            except Exception as e:
                if idx == 0:
                    print(f"      Error in first spectrum: {e}")
                continue
        
        print(f"  ✓ Extracted {len(ms1_scans)} MS1, {len(ms2_scans)} MS2")
        if ms1_times:
            print(f"  ✓ Retention time range: {min(ms1_times):.3f} - {max(ms1_times):.3f} min")
    
    except Exception as e:
        print(f"  ✗ Error: {e}")
        import traceback
        traceback.print_exc()
        return None, None, None
    
    return ms1_scans, ms1_times, ms2_scans

# ============================================================================
# FUNCTION: Generate High-Quality MS1 Plot
# ============================================================================

def generate_ms1_plot(filepath, ms1_scans, ms1_times):
    """Generate a large, high-quality MS1 spectrum plot showing peak distributions."""
    
    if len(ms1_scans) == 0:
        return None
    
    clean_name = get_clean_filename(filepath)
    output_file = Path(filepath).parent / f"{clean_name}_MS1_Spectrum.png"
    
    first_scan = ms1_scans[0]
    mz = first_scan['mz']
    intensity = first_scan['intensity']
    
    fig, ax = plt.subplots(figsize=(20, 10))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    
    ax.plot(mz, intensity, linewidth=1.5, color='#1f77b4')
    ax.fill_between(mz, intensity, alpha=0.35, color='#1f77b4')
    
    ax.set_xlabel('m/z', fontsize=24, fontweight='bold', labelpad=15)
    ax.set_ylabel('Intensity', fontsize=24, fontweight='bold', labelpad=15)
    
    ax.tick_params(axis='both', which='major', labelsize=18)
    ax.tick_params(axis='both', which='major', width=2, length=8)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(2)
    ax.spines['bottom'].set_linewidth(2)
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"  ✓ Saved MS1 plot: {output_file.name}")
    return output_file

# ============================================================================
# FUNCTION: Generate TIC Plot
# ============================================================================

def generate_tic_plot(filepath, ms1_scans, ms1_times):
    """Generate a high-quality TIC plot (full range 0-10 min)."""
    
    if len(ms1_scans) == 0 or len(ms1_times) == 0:
        return None
    
    clean_name = get_clean_filename(filepath)
    output_file = Path(filepath).parent / f"{clean_name}_TIC.png"
    
    tic = [np.sum(scan['intensity']) for scan in ms1_scans]
    
    fig, ax = plt.subplots(figsize=(20, 8))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    
    ax.plot(ms1_times, tic, linewidth=2, color='#2ca02c')
    ax.fill_between(ms1_times, tic, alpha=0.3, color='#2ca02c')
    
    ax.set_xlabel('Retention Time (min)', fontsize=24, fontweight='bold', labelpad=15)
    ax.set_ylabel('Total Ion Current', fontsize=24, fontweight='bold', labelpad=15)
    
    ax.tick_params(axis='both', which='major', labelsize=18)
    ax.tick_params(axis='both', which='major', width=2, length=8)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(2)
    ax.spines['bottom'].set_linewidth(2)
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"  ✓ Saved TIC plot: {output_file.name}")
    return output_file

# ============================================================================
# FUNCTION: Generate TIC SHORT Plot (3-9 min, narrow format, large fonts)
# ============================================================================

def generate_tic_short_plot(filepath, ms1_scans, ms1_times):
    """Generate TIC plot showing only 3-9 min window in narrow format with large fonts."""
    
    if len(ms1_scans) == 0 or len(ms1_times) == 0:
        return None
    
    clean_name = get_clean_filename(filepath)
    output_file = Path(filepath).parent / f"{clean_name}_TIC_SHORT.png"
    
    tic = [np.sum(scan['intensity']) for scan in ms1_scans]
    
    # Filter data for 3-9 min range
    mask = (np.array(ms1_times) >= 3.0) & (np.array(ms1_times) <= 9.0)
    filtered_times = np.array(ms1_times)[mask]
    filtered_tic = np.array(tic)[mask]
    
    if len(filtered_times) == 0:
        print(f"  ⚠ No data in 3-9 min range for TIC SHORT")
        return None
    
    # Narrow figure format
    fig, ax = plt.subplots(figsize=(14, 8))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    
    ax.plot(filtered_times, filtered_tic, linewidth=3, color='#2ca02c')
    ax.fill_between(filtered_times, filtered_tic, alpha=0.3, color='#2ca02c')
    
    ax.set_xlabel('Retention Time (min)', fontsize=26, fontweight='bold', labelpad=15)
    ax.set_ylabel('Total Ion Current', fontsize=26, fontweight='bold', labelpad=15)
    
    ax.tick_params(axis='both', which='major', labelsize=20)
    ax.tick_params(axis='both', which='major', width=2.5, length=10)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(2.5)
    ax.spines['bottom'].set_linewidth(2.5)
    
    # Set x-axis limits
    ax.set_xlim(3.0, 9.0)
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"  ✓ Saved TIC SHORT plot: {output_file.name}")
    return output_file

# ============================================================================
# FUNCTION: Generate BPC Plot
# ============================================================================

def generate_bpc_plot(filepath, ms1_scans, ms1_times):
    """Generate a high-quality BPC plot (full range 0-10 min)."""
    
    if len(ms1_scans) == 0 or len(ms1_times) == 0:
        return None
    
    clean_name = get_clean_filename(filepath)
    output_file = Path(filepath).parent / f"{clean_name}_BPC.png"
    
    bpc = [np.max(scan['intensity']) for scan in ms1_scans]
    
    fig, ax = plt.subplots(figsize=(20, 8))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    
    ax.plot(ms1_times, bpc, linewidth=2.5, color='#d62728')
    ax.fill_between(ms1_times, bpc, alpha=0.3, color='#d62728')
    
    ax.set_xlabel('Retention Time (min)', fontsize=24, fontweight='bold', labelpad=15)
    ax.set_ylabel('Base Peak Intensity', fontsize=24, fontweight='bold', labelpad=15)
    
    ax.tick_params(axis='both', which='major', labelsize=18)
    ax.tick_params(axis='both', which='major', width=2, length=8)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(2)
    ax.spines['bottom'].set_linewidth(2)
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"  ✓ Saved BPC plot: {output_file.name}")
    return output_file

# ============================================================================
# FUNCTION: Generate BPC SHORT Plot (3-9 min, narrow format, large fonts)
# ============================================================================

def generate_bpc_short_plot(filepath, ms1_scans, ms1_times):
    """Generate BPC plot showing only 3-9 min window in narrow format with large fonts."""
    
    if len(ms1_scans) == 0 or len(ms1_times) == 0:
        return None
    
    clean_name = get_clean_filename(filepath)
    output_file = Path(filepath).parent / f"{clean_name}_BPC_SHORT.png"
    
    bpc = [np.max(scan['intensity']) for scan in ms1_scans]
    
    # Filter data for 3-9 min range
    mask = (np.array(ms1_times) >= 3.0) & (np.array(ms1_times) <= 9.0)
    filtered_times = np.array(ms1_times)[mask]
    filtered_bpc = np.array(bpc)[mask]
    
    if len(filtered_times) == 0:
        print(f"  ⚠ No data in 3-9 min range for BPC SHORT")
        return None
    
    # Narrow figure format
    fig, ax = plt.subplots(figsize=(14, 8))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    
    ax.plot(filtered_times, filtered_bpc, linewidth=3, color='#d62728')
    ax.fill_between(filtered_times, filtered_bpc, alpha=0.3, color='#d62728')
    
    ax.set_xlabel('Retention Time (min)', fontsize=26, fontweight='bold', labelpad=15)
    ax.set_ylabel('Base Peak Intensity', fontsize=26, fontweight='bold', labelpad=15)
    
    ax.tick_params(axis='both', which='major', labelsize=20)
    ax.tick_params(axis='both', which='major', width=2.5, length=10)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(2.5)
    ax.spines['bottom'].set_linewidth(2.5)
    
    # Set x-axis limits
    ax.set_xlim(3.0, 9.0)
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"  ✓ Saved BPC SHORT plot: {output_file.name}")
    return output_file

# ============================================================================
# UPDATED: Generate All Plots Function
# ============================================================================

def generate_plots(filepath, ms1_scans, ms1_times, ms2_scans):
    """Generate all plots including full and SHORT versions."""
    
    if len(ms1_scans) == 0 and len(ms2_scans) == 0:
        print(f"  ✗ No data to plot")
        return None
    
    # Generate full range plots
    generate_ms1_plot(filepath, ms1_scans, ms1_times)
    generate_tic_plot(filepath, ms1_scans, ms1_times)
    generate_bpc_plot(filepath, ms1_scans, ms1_times)
    
    # Generate SHORT versions (3-9 min, narrow format, large fonts)
    generate_tic_short_plot(filepath, ms1_scans, ms1_times)
    generate_bpc_short_plot(filepath, ms1_scans, ms1_times)
    
    return True

def process_all_files():
    """Process all .mzdata.xml files in directory."""
    
    input_path = Path(INPUT_DIRECTORY)
    
    if not input_path.exists():
        print(f"✗ Directory not found: {INPUT_DIRECTORY}")
        return
    
    files = sorted(list(input_path.glob('*.mzdata.xml')))
    
    if not files:
        print(f"✗ No *.mzdata.xml files found in:\n  {INPUT_DIRECTORY}")
        return
    
    print(f"\n{'='*70}")
    print(f"Processing {len(files)} mzData files")
    print(f"Retention time calculation: 900 scans = 10 minutes")
    print(f"{'='*70}")
    
    success = 0
    failed = 0
    
    for filepath in files:
        ms1, times, ms2 = read_mzdata_file(filepath)
        
        if ms1 is None:
            failed += 1
            continue
        
        if generate_plots(filepath, ms1, times, ms2):
            success += 1
        else:
            failed += 1
    
    print(f"\n{'='*70}")
    print(f"Complete! Success: {success}/{len(files)}")
    if failed > 0:
        print(f"Failed: {failed}/{len(files)}")
    print(f"{'='*70}\n")

if __name__ == "__main__":
    process_all_files()

In [ ]:
# batch export interactive LCMS map where you can view and annotate MS1 data in html 

import numpy as np
import plotly.graph_objects as go
from pathlib import Path
import xml.etree.ElementTree as ET
import base64
import zlib
import json

# ============================================================================
# CONFIGURATION
# ============================================================================

# Place your data in folders relative to this script
INPUT_FOLDER = "./data/input"  # Folder with .mzdata.xml files

# ============================================================================
# FUNCTION 1: Decode Binary Data
# ============================================================================

def decode_binary_data(encoded_data, precision=32, endian='little', compression=None):
    """Decode base64 encoded binary data."""
    try:
        binary = base64.b64decode(encoded_data)
        
        if compression == 'zlib':
            binary = zlib.decompress(binary)
        
        endian_char = '<' if endian == 'little' else '>'
        dtype = f'{endian_char}f8' if precision == 64 else f'{endian_char}f4'
        data = np.frombuffer(binary, dtype=dtype)
        return data
    except:
        return np.array([])

# ============================================================================
# FUNCTION 2: Read mzData File
# ============================================================================

def read_mzdata_file(filepath):
    """Read mzData XML and extract MS1 scans."""
    ms1_scans = []
    ms1_times = []
    
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()
        
        spectra = root.findall('.//{*}spectrum')
        
        # Calculate retention time per spectrum
        time_per_scan = 10.0 / 900.0  # 900 scans = 10 minutes
        
        for idx, spectrum in enumerate(spectra):
            try:
                # Get MS level
                ms_level = '1'
                spec_instr = spectrum.find('.//{*}spectrumInstrument')
                if spec_instr is not None:
                    ms_level = spec_instr.get('msLevel', '1')
                
                if ms_level != '1':
                    continue
                
                # Calculate retention time
                rt = idx * time_per_scan
                
                # Find mz and intensity arrays
                mz_array_binary = spectrum.find('{*}mzArrayBinary')
                intensity_array_binary = spectrum.find('{*}intenArrayBinary')
                
                if mz_array_binary is None or intensity_array_binary is None:
                    continue
                
                # Extract data elements
                mz_data_elem = mz_array_binary.find('{*}data')
                if mz_data_elem is None:
                    mz_data_elem = mz_array_binary.find('data')
                
                intensity_data_elem = intensity_array_binary.find('{*}data')
                if intensity_data_elem is None:
                    intensity_data_elem = intensity_array_binary.find('data')
                
                if mz_data_elem is None or intensity_data_elem is None:
                    continue
                
                # Get precision and encoding
                mz_precision = int(mz_data_elem.get('precision', 32))
                mz_endian = mz_data_elem.get('endian', 'little')
                mz_compression = mz_data_elem.get('compression', None)
                
                intensity_precision = int(intensity_data_elem.get('precision', 32))
                intensity_endian = intensity_data_elem.get('endian', 'little')
                intensity_compression = intensity_data_elem.get('compression', None)
                
                # Decode data
                mz_data = decode_binary_data(
                    mz_data_elem.text or '',
                    mz_precision,
                    mz_endian,
                    mz_compression
                )
                
                intensity_data = decode_binary_data(
                    intensity_data_elem.text or '',
                    intensity_precision,
                    intensity_endian,
                    intensity_compression
                )
                
                if len(mz_data) > 0 and len(intensity_data) > 0:
                    ms1_scans.append({
                        'mz': mz_data.astype(float),
                        'intensity': intensity_data.astype(float),
                    })
                    ms1_times.append(float(rt))
            
            except:
                continue
    
    except Exception as e:
        print(f"  ✗ Error reading {filepath.name}: {e}")
        return None, None
    
    return ms1_scans, ms1_times

# ============================================================================
# FUNCTION 3: Create Interactive HTML Plot (WITH GREY ANNOTATIONS)
# ============================================================================

def create_interactive_html_plot(mzdata_file):
    """Create interactive HTML plot with grey peak annotations."""
    
    ms1_scans, ms1_times = read_mzdata_file(mzdata_file)
    
    if ms1_scans is None or len(ms1_scans) == 0:
        return None
    
    # Calculate TIC
    tic = [float(np.sum(scan['intensity'])) for scan in ms1_scans]
    
    # ========== PREPARE MS1 DATA FOR JSON ==========
    ms1_data_json = json.dumps([
        {
            'mz': [float(x) for x in scan['mz']],
            'intensity': [float(y) for y in scan['intensity']],
        }
        for scan in ms1_scans
    ])
    
    ms1_times_json = json.dumps([float(x) for x in ms1_times])
    
    # ========== CREATE CUSTOM HTML WITH GREY ANNOTATIONS ==========
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>Interactive TIC and MS1 Explorer with Annotations</title>
        <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
        <style>
            * {{
                margin: 0;
                padding: 0;
                box-sizing: border-box;
            }}
            body {{
                font-family: Arial, sans-serif;
                background-color: #f5f5f5;
            }}
            .container {{
                max-width: 1800px;
                margin: 0 auto;
                padding: 20px;
                display: grid;
                grid-template-columns: 1fr 1fr;
                gap: 20px;
                grid-auto-rows: max-content;
            }}
            .card {{
                background-color: white;
                border-radius: 8px;
                box-shadow: 0 2px 10px rgba(0,0,0,0.1);
                padding: 15px;
            }}
            h1 {{
                grid-column: 1 / -1;
                text-align: center;
                color: #333;
                margin-bottom: 10px;
            }}
            .info {{
                grid-column: 1 / -1;
                text-align: center;
                color: #666;
                font-size: 13px;
                margin-bottom: 15px;
            }}
            .instructions {{
                grid-column: 1 / -1;
                background-color: #e8f4f8;
                border-left: 4px solid #2ca02c;
                padding: 15px;
                margin-bottom: 15px;
                border-radius: 4px;
            }}
            .instructions h3 {{
                color: #2ca02c;
                margin-bottom: 10px;
                font-size: 14px;
            }}
            .instructions ul {{
                margin-left: 20px;
                font-size: 13px;
            }}
            .instructions li {{
                margin: 5px 0;
            }}
            #ticPlot {{
                width: 100%;
                height: 500px;
                cursor: pointer;
            }}
            #ms1Plot {{
                width: 100%;
                height: 500px;
                cursor: crosshair;
            }}
            .plot-title {{
                font-weight: bold;
                font-size: 14px;
                margin-bottom: 10px;
                color: #333;
                display: flex;
                justify-content: space-between;
                align-items: center;
            }}
            .button-group {{
                display: flex;
                gap: 8px;
            }}
            .btn {{
                padding: 6px 12px;
                font-size: 12px;
                border: none;
                border-radius: 4px;
                cursor: pointer;
                font-weight: bold;
                transition: all 0.3s;
            }}
            .btn-clear {{
                background-color: #ff7f0e;
                color: white;
            }}
            .btn-clear:hover {{
                background-color: #ff6b00;
            }}
            .btn-clear:disabled {{
                background-color: #ccc;
                cursor: not-allowed;
            }}
            .annotations-list {{
                margin-top: 15px;
                padding-top: 15px;
                border-top: 1px solid #eee;
                max-height: 300px;
                overflow-y: auto;
            }}
            .annotation-item {{
                display: flex;
                justify-content: space-between;
                align-items: center;
                padding: 8px;
                margin-bottom: 5px;
                background-color: #f9f9f9;
                border-left: 3px solid #888;
                border-radius: 3px;
                font-size: 12px;
            }}
            .annotation-value {{
                font-weight: bold;
                color: #333;
            }}
            .btn-delete {{
                background-color: #d62728;
                color: white;
                padding: 3px 8px;
                font-size: 11px;
            }}
            .btn-delete:hover {{
                background-color: #a01a1a;
            }}
            .status {{
                grid-column: 1 / -1;
                text-align: center;
                color: #2ca02c;
                font-weight: bold;
                margin-top: 10px;
                font-size: 13px;
            }}
            .hint {{
                grid-column: 2 / 3;
                color: #999;
                font-size: 12px;
                text-align: center;
                font-style: italic;
            }}
        </style>
    </head>
    <body>
        <div class="container">
            <h1>Interactive TIC and MS1 Explorer with Peak Annotations</h1>
            
            <div class="info">
                File: <b>{Path(mzdata_file).name}</b> | 
                Scans: <b>{len(ms1_scans)}</b> | 
                RT Range: <b>{min(ms1_times):.2f} - {max(ms1_times):.2f}</b> min | 
                TIC Range: <b>{min(tic):.2e} - {max(tic):.2e}</b>
            </div>
            
            <div class="instructions">
                <h3>📊 How to Use:</h3>
                <ul>
                    <li><b>Left (TIC):</b> Click any point to view its MS1 spectrum on the right</li>
                    <li><b>Right (MS1):</b> Click on any peak to annotate it with m/z value</li>
                    <li><b>Annotations:</b> Appear as grey labels with thin lines</li>
                    <li><b>Delete:</b> Click delete button in annotation list or clear all</li>
                    <li><b>Zoom/Pan:</b> Use standard Plotly controls</li>
                </ul>
            </div>
            
            <div class="card">
                <div class="plot-title">Total Ion Chromatogram (TIC)</div>
                <div id="ticPlot"></div>
            </div>
            
            <div class="card">
                <div class="plot-title">
                    <span>MS1 Spectrum (Scan 0) - Click peaks to annotate</span>
                    <div class="button-group">
                        <button class="btn btn-clear" id="clearAnnotationsBtn" onclick="clearAllAnnotations()">
                            Clear All
                        </button>
                    </div>
                </div>
                <div id="ms1Plot"></div>
                <div class="annotations-list" id="annotationsList">
                    <div style="text-align: center; color: #999; font-size: 12px;">No annotations yet</div>
                </div>
            </div>
            
            <div class="hint">💡 Click on peaks in the MS1 spectrum to add annotations. Hover over them to see details.</div>
            
            <div class="status">✓ Ready - Click on TIC to view MS1, then click peaks to annotate</div>
        </div>

        <script>
            console.log('Script starting...');
            
            // Load data
            const ms1Scans = {ms1_data_json};
            const ms1Times = {ms1_times_json};
            
            console.log('Loaded', ms1Scans.length, 'MS1 scans');
            console.log('Loaded', ms1Times.length, 'time points');
            
            // Annotation management
            let annotations = [];
            let currentScanIndex = 0;
            
            // ========== PLOT MS1 WITH ANNOTATIONS ==========
            function plotMS1(scanIndex) {{
                console.log('Plotting MS1 for scan', scanIndex);
                currentScanIndex = scanIndex;
                const scan = ms1Scans[scanIndex];
                const rt = ms1Times[scanIndex];
                
                if (!scan || !scan.mz || !scan.intensity) {{
                    console.error('Invalid scan data:', scan);
                    return;
                }}
                
                const ms1Trace = {{
                    x: scan.mz,
                    y: scan.intensity,
                    mode: 'lines',
                    name: 'MS1',
                    line: {{color: '#1f77b4', width: 1.5}},
                    fill: 'tozeroy',
                    fillcolor: 'rgba(31, 119, 180, 0.2)',
                    hovertemplate: '<b>m/z:</b> %{{x:.4f}}<br><b>Intensity:</b> %{{y:.2e}}<extra></extra>'
                }};
                
                // Build annotations for this scan (GREY ANNOTATIONS)
                const plotAnnotations = annotations
                    .filter(a => a.scanIndex === scanIndex)
                    .map(a => ({{
                        x: a.mz,
                        y: a.intensity,
                        text: a.mz.toFixed(4),
                        showarrow: true,
                        arrowhead: 0,
                        arrowsize: 0.5,
                        arrowwidth: 1,
                        arrowcolor: '#888888',
                        ax: 0,
                        ay: -40,
                        xanchor: 'center',
                        yanchor: 'top',
                        bgcolor: 'transparent',
                        bordercolor: 'transparent',
                        font: {{color: '#888888', size: 11}},
                    }}));
                
                const ms1Layout = {{
                    title: `Scan #${{scanIndex}} - RT: ${{rt.toFixed(3)}} min | Annotations: ${{annotations.filter(a => a.scanIndex === scanIndex).length}}`,
                    xaxis: {{title: 'm/z', showgrid: true, gridwidth: 1, gridcolor: 'LightGray'}},
                    yaxis: {{title: 'Intensity', showgrid: true, gridwidth: 1, gridcolor: 'LightGray'}},
                    hovermode: 'x unified',
                    template: 'plotly_white',
                    margin: {{l: 60, r: 20, t: 60, b: 60}},
                    plot_bgcolor: 'white',
                    paper_bgcolor: 'white',
                    annotations: plotAnnotations,
                }};
                
                Plotly.newPlot('ms1Plot', [ms1Trace], ms1Layout, {{responsive: true}});
                
                // Add click handler after plotting
                document.getElementById('ms1Plot').on('plotly_click', onMS1Click);
                
                updateAnnotationsList();
                console.log('MS1 plot updated for scan', scanIndex);
            }}
            
            // ========== ADD ANNOTATION ON MS1 CLICK ==========
            function onMS1Click(data) {{
                const point = data.points[0];
                const mz = point.x;
                const intensity = point.y;
                
                console.log('MS1 Click detected at m/z =', mz.toFixed(4));
                
                // Check if this peak is already annotated
                const exists = annotations.some(a => 
                    a.scanIndex === currentScanIndex && 
                    Math.abs(a.mz - mz) < 0.01  // Within 0.01 m/z tolerance
                );
                
                if (exists) {{
                    console.log(`Peak at m/z ${{mz.toFixed(4)}} already annotated`);
                    alert(`Peak m/z ${{mz.toFixed(4)}} is already annotated!`);
                    return;
                }}
                
                // Add annotation
                annotations.push({{
                    scanIndex: currentScanIndex,
                    mz: mz,
                    intensity: intensity,
                    id: Date.now()
                }});
                
                console.log(`Added annotation at m/z=${{mz.toFixed(4)}}`);
                
                // Refresh plot
                plotMS1(currentScanIndex);
            }}
            
            // ========== UPDATE ANNOTATIONS LIST ==========
            function updateAnnotationsList() {{
                const list = document.getElementById('annotationsList');
                const currentAnnotations = annotations.filter(a => a.scanIndex === currentScanIndex);
                
                if (currentAnnotations.length === 0) {{
                    list.innerHTML = '<div style="text-align: center; color: #999; font-size: 12px;">No annotations yet</div>';
                    return;
                }}
                
                list.innerHTML = currentAnnotations.map((ann, idx) => `
                    <div class="annotation-item">
                        <span>
                            <b>#${{idx + 1}}</b>: 
                            <span class="annotation-value">m/z ${{ann.mz.toFixed(4)}}</span>
                            (Intensity: ${{ann.intensity.toExponential(2)}})
                        </span>
                        <button class="btn btn-delete" onclick="deleteAnnotation(${{ann.id}})">✕ Delete</button>
                    </div>
                `).join('');
            }}
            
            // ========== DELETE SINGLE ANNOTATION ==========
            function deleteAnnotation(id) {{
                annotations = annotations.filter(a => a.id !== id);
                plotMS1(currentScanIndex);
            }}
            
            // ========== CLEAR ALL ANNOTATIONS ==========
            function clearAllAnnotations() {{
                if (annotations.length === 0) {{
                    alert('No annotations to clear');
                    return;
                }}
                
                if (confirm(`Delete all ${{annotations.length}} annotations?`)) {{
                    annotations = [];
                    plotMS1(currentScanIndex);
                    console.log('All annotations cleared');
                }}
            }}
            
            // ========== TIC PLOT ==========
            const ticTrace = {{
                x: ms1Times,
                y: ms1Scans.map(s => s.intensity.reduce((a, b) => a + b, 0)),
                mode: 'lines+markers',
                name: 'TIC',
                line: {{color: '#2ca02c', width: 2}},
                marker: {{size: 4, color: '#2ca02c'}},
                fill: 'tozeroy',
                fillcolor: 'rgba(44, 160, 44, 0.2)',
                hovertemplate: '<b>RT:</b> %{{x:.3f}} min<br><b>TIC:</b> %{{y:.2e}}<extra></extra>'
            }};
            
            const ticLayout = {{
                title: 'Click on a point to view its MS1 spectrum',
                xaxis: {{title: 'Retention Time (min)', showgrid: true, gridwidth: 1, gridcolor: 'LightGray'}},
                yaxis: {{title: 'Total Ion Current', showgrid: true, gridwidth: 1, gridcolor: 'LightGray'}},
                hovermode: 'x unified',
                template: 'plotly_white',
                margin: {{l: 60, r: 20, t: 40, b: 60}},
                plot_bgcolor: 'white',
                paper_bgcolor: 'white',
            }};
            
            console.log('Creating TIC plot...');
            Plotly.newPlot('ticPlot', [ticTrace], ticLayout, {{responsive: true}});
            
            // ========== TIC CLICK HANDLER ==========
            document.getElementById('ticPlot').on('plotly_click', function(data) {{
                console.log('TIC click detected');
                const point = data.points[0];
                const rtClicked = point.x;
                
                // Find closest scan
                let closestIndex = 0;
                let minDist = Math.abs(ms1Times[0] - rtClicked);
                
                for (let i = 1; i < ms1Times.length; i++) {{
                    const dist = Math.abs(ms1Times[i] - rtClicked);
                    if (dist < minDist) {{
                        minDist = dist;
                        closestIndex = i;
                    }}
                }}
                
                console.log('Closest scan:', closestIndex, 'at RT:', ms1Times[closestIndex].toFixed(3));
                
                // Update MS1 plot
                plotMS1(closestIndex);
                
                // Update TIC title
                Plotly.relayout('ticPlot', {{'title': `Selected: Scan #${{closestIndex}} at RT=${{ms1Times[closestIndex].toFixed(3)}} min`}});
            }});
            
            // Initial plot
            console.log('Creating initial MS1 plot for scan 0...');
            plotMS1(0);
            
            console.log('Script initialization complete!');
        </script>
    </body>
    </html>
    """
    
    # Save HTML file in SAME FOLDER as mzdata file
    output_html = mzdata_file.parent / f"{mzdata_file.stem}_explorer.html"
    
    try:
        with open(output_html, 'w') as f:
            f.write(html_content)
        return str(output_html)
    except Exception as e:
        print(f"  ✗ Error saving HTML: {e}")
        return None

# ============================================================================
# FUNCTION 4: Batch Process All mzdata Files in Folder
# ============================================================================

def batch_process_folder(input_folder):
    """Process all .mzdata.xml files in a folder and create HTML explorers."""
    
    input_path = Path(input_folder)
    
    if not input_path.exists():
        print(f"✗ Folder not found: {input_folder}")
        return
    
    # Find all mzdata files
    mzdata_files = sorted(list(input_path.glob('*.mzdata.xml')))
    
    if not mzdata_files:
        print(f"✗ No *.mzdata.xml files found in: {input_folder}")
        return
    
    print(f"\n{'='*70}")
    print(f"Batch Processing {len(mzdata_files)} mzData Files")
    print(f"{'='*70}\n")
    
    success = 0
    failed = 0
    
    for idx, filepath in enumerate(mzdata_files, 1):
        print(f"[{idx}/{len(mzdata_files)}] Processing: {filepath.name}...", end=' ')
        
        try:
            output_html = create_interactive_html_plot(filepath)
            
            if output_html:
                print(f"✓ Created")
                print(f"           → {Path(output_html).name}")
                success += 1
            else:
                print(f"✗ Failed (no data)")
                failed += 1
        
        except Exception as e:
            print(f"✗ Error: {e}")
            failed += 1
    
    print(f"\n{'='*70}")
    print(f"Batch Processing Complete!")
    print(f"{'='*70}")
    print(f"  ✓ Success: {success}/{len(mzdata_files)}")
    print(f"  ✗ Failed:  {failed}/{len(mzdata_files)}")
    print(f"\nAll HTML files saved in their respective directories")
    print(f"{'='*70}\n")

# ============================================================================
# RUN BATCH PROCESSING
# ============================================================================

batch_process_folder(INPUT_FOLDER)